# DSA 4060 – Recommender Systems

## Week 2 Practical: Recommendation Data and Popularity Baselines

### Objective

- The purpose of this practical is to explore movie-rating interaction data and build a simple popularity-based recommender system.

- The analysis will involve cleaning the ratings dataset, exploring user–movie interactions, creating a user-item interaction matrix, calculating dataset sparsity, and recommending the top 10 movies based on their average ratings and number of ratings.

In [1]:
import pandas as pd
import numpy as np

## 1. Load and Inspect the Dataset

The first step is to load the movie ratings dataset using pandas. The dataset contains information about users, movies, ratings, and the time when each rating was recorded.

In [21]:
df = pd.read_csv("/Users/abdiqalaqmohamed/Desktop/dsa4060-week2-popularity-recommender/data/DSA_4060_Week2_Movie_Ratings.csv")

df.head(10)

,user_id,movie_id,movie_title,rating,timestamp
0,U001,M001,Nairobi Nights,3,2026-01-01T08:00:00Z
1,U004,M001,Nairobi Nights,4,2026-01-06T09:00:00Z
2,U007,M001,Nairobi Nights,2,2026-01-11T10:00:00Z
3,U010,M001,Nairobi Nights,2,2026-01-16T11:00:00Z
4,U013,M001,Nairobi Nights,4,2026-01-21T12:00:00Z
5,U016,M001,Nairobi Nights,3,2026-01-26T13:00:00Z
6,U019,M001,Nairobi Nights,3,2026-01-31T14:00:00Z
7,U022,M001,Nairobi Nights,2,2026-02-05T15:00:00Z
8,U025,M001,Nairobi Nights,2,2026-02-10T16:00:00Z
9,U028,M001,Nairobi Nights,4,2026-02-15T17:00:00Z


### Dataset Dimensions and Data Types

Next, I examine the size of the dataset and the data types of each column. This helps to understand the structure of the data before performing any cleaning or analysis.

In [4]:
print("Dataset dimensions:", df.shape)

print("\nData types:")
print(df.dtypes)


Dataset dimensions: (509, 5)

Data types:
user_id          str
movie_id         str
movie_title      str
rating         int64
timestamp        str
dtype: object


### Missing Values

Missing values can affect the accuracy of the analysis and recommendation system. I therefore check each column to determine whether any values are missing.

In [5]:
df.isnull().sum()

user_id        2
movie_id       1
movie_title    1
rating         0
timestamp      0
dtype: int64

### Duplicate Records

Duplicate records may cause some user interactions to be counted more than once. I check the dataset for duplicated rows before performing the analysis.

In [6]:
duplicate_count = df.duplicated().sum()

print("Number of duplicate records:", duplicate_count)

Number of duplicate records: 3


### Initial Inspection Summary

The dataset contains 509 records and 5 columns. In addition to the user ID, movie ID, rating, and timestamp, the dataset also contains the movie title.

The initial inspection identified missing values in the user ID, movie ID, and movie title columns. Three duplicate records were also found. These issues will be handled during the data-cleaning stage before building the recommender system.

## 2. Data Cleaning

Before analysing the interaction data, the dataset needs to be cleaned. This involves removing duplicate records, removing records with missing user or movie IDs, and checking whether the ratings fall within the expected rating scale.

### Remove Duplicate Records

Duplicate records can cause the same interaction to be counted more than once. I remove duplicate rows to ensure that each recorded interaction is only included once.

In [7]:
df_clean = df.drop_duplicates().copy()

print("Rows before removing duplicates:", len(df))
print("Rows after removing duplicates:", len(df_clean))
print("Duplicates remaining:", df_clean.duplicated().sum())

Rows before removing duplicates: 509
Rows after removing duplicates: 506
Duplicates remaining: 0


### Remove Missing User and Movie IDs

User and movie IDs are essential for creating the user-item interaction matrix. Records without either of these identifiers cannot be used by the recommender system, so they are removed.

In [8]:
df_clean = df_clean.dropna(subset=["user_id", "movie_id"]).copy()

print("Rows remaining:", len(df_clean))

print("\nMissing values after removing missing IDs:")
print(df_clean.isnull().sum())

Rows remaining: 503

Missing values after removing missing IDs:
user_id        0
movie_id       0
movie_title    0
rating         0
timestamp      0
dtype: int64


### Check the Rating Scale

The expected rating scale is from 1 to 5. I check the minimum and maximum ratings and identify any records that fall outside this range.

In [9]:
print("Minimum rating:", df_clean["rating"].min())
print("Maximum rating:", df_clean["rating"].max())

invalid_ratings = df_clean[
    ~df_clean["rating"].between(1, 5)
]

print("\nNumber of invalid ratings:", len(invalid_ratings))

invalid_ratings

Minimum rating: -1
Maximum rating: 6

Number of invalid ratings: 3


,user_id,movie_id,movie_title,rating,timestamp
506,U006,M010,Beyond the Horizon,0,2026-08-18T09:00:00Z
507,U028,M015,The Green Valley,6,2026-08-19T13:00:00Z
508,U041,M023,The Startup,-1,2026-08-20T14:00:00Z


### Remove Invalid Ratings

The rating scale is expected to range from 1 to 5. Three records were found outside this range, with ratings of 0, 6, and -1. Since these values are invalid, I remove these records rather than altering their ratings.

In [10]:
df_clean = df_clean[
    df_clean["rating"].between(1, 5)
].copy()

print("Rows after cleaning:", len(df_clean))
print("Minimum rating:", df_clean["rating"].min())
print("Maximum rating:", df_clean["rating"].max())

Rows after cleaning: 500
Minimum rating: 1
Maximum rating: 5


### Cleaning Verification

After completing the cleaning process, I verify that there are no duplicate records, no missing user or movie IDs, and no ratings outside the valid 1–5 range.

In [11]:
print("Cleaned dataset dimensions:", df_clean.shape)
print("Duplicate records:", df_clean.duplicated().sum())
print("Missing user IDs:", df_clean["user_id"].isnull().sum())
print("Missing movie IDs:", df_clean["movie_id"].isnull().sum())
print(
    "Invalid ratings:",
    (~df_clean["rating"].between(1, 5)).sum()
)

Cleaned dataset dimensions: (500, 5)
Duplicate records: 0
Missing user IDs: 0
Missing movie IDs: 0
Invalid ratings: 0


### Cleaning Summary

The original dataset contained 509 records. Three duplicate records were removed, records with missing user or movie IDs were excluded, and three ratings outside the valid 1–5 scale were removed. After cleaning, the dataset contains 500 valid user-movie rating interactions that can be used for further analysis.

## 3. Exploratory Analysis of User-Movie Interactions

In this section, I explore the cleaned dataset to understand the interaction patterns between users and movies. I calculate the total number of ratings, number of unique users and movies, the most active user, the most-rated movie, and the overall average rating.

In [12]:
total_ratings = len(df_clean)
unique_users = df_clean["user_id"].nunique()
unique_movies = df_clean["movie_id"].nunique()
average_rating = df_clean["rating"].mean()

print("Total number of ratings:", total_ratings)
print("Number of unique users:", unique_users)
print("Number of unique movies:", unique_movies)
print("Average rating:", round(average_rating, 2))

Total number of ratings: 500
Number of unique users: 50
Number of unique movies: 24
Average rating: 3.45


### Most Active User

The most active user is the user who has submitted the highest number of movie ratings. This provides an indication of which user has interacted most frequently with the system.

In [13]:
user_rating_counts = df_clean["user_id"].value_counts()

most_active_user = user_rating_counts.idxmax()
most_active_count = user_rating_counts.max()

print("Most active user:", most_active_user)
print("Number of ratings:", most_active_count)

Most active user: U010
Number of ratings: 17


### Most-Rated Movie

The most-rated movie is the movie that has received the largest number of user ratings. Unlike the highest-rated movie, this measures the amount of interaction with the movie rather than how positively users rated it.

In [14]:
movie_rating_counts = df_clean.groupby(
    ["movie_id", "movie_title"]
).size().sort_values(ascending=False)

most_rated_movie = movie_rating_counts.index[0]
most_rated_count = movie_rating_counts.iloc[0]

print("Most-rated movie ID:", most_rated_movie[0])
print("Most-rated movie:", most_rated_movie[1])
print("Number of ratings:", most_rated_count)

Most-rated movie ID: M001
Most-rated movie: Nairobi Nights
Number of ratings: 37


### Exploratory Analysis Summary

After cleaning, the dataset contains 500 ratings from 50 unique users across 24 unique movies. The average rating is approximately 3.45 out of 5.

User U010 is the most active user with 17 ratings, while *Nairobi Nights* (M001) is the most-rated movie with 37 ratings. These results show that the level of interaction varies across both users and movies.

## 4. User-Item Interaction Matrix

A user-item interaction matrix represents the relationship between users and movies. Each row represents a user, each column represents a movie, and each cell contains the rating given by a user to a movie.

If a user has not rated a particular movie, the corresponding cell will contain a missing value (NaN).

In [15]:
user_item_matrix = df_clean.pivot_table(
    index="user_id",
    columns="movie_id",
    values="rating",
    aggfunc="mean"
)

print("User-item matrix dimensions:", user_item_matrix.shape)

user_item_matrix.iloc[:10, :10]

User-item matrix dimensions: (50, 24)


movie_id,M001,M002,M003,M004,M005,M006,M007,M008,M009,M010
user_id,,,,,,,,,,
U001,3.000000,2.000000,NaN,NaN,4.0,4.0,NaN,NaN,NaN,NaN
U002,3.000000,NaN,NaN,3.5,NaN,NaN,3.0,NaN,NaN,NaN
U003,NaN,NaN,NaN,NaN,4.0,NaN,NaN,4.0,4.0,NaN
U004,4.000000,NaN,NaN,NaN,3.0,3.0,NaN,NaN,NaN,NaN
U005,2.000000,4.000000,NaN,3.0,NaN,NaN,5.0,NaN,NaN,NaN
U006,3.000000,3.000000,NaN,5.0,4.0,NaN,NaN,3.0,NaN,NaN
U007,2.333333,NaN,NaN,NaN,5.0,4.0,NaN,NaN,2.0,NaN
U008,3.000000,2.000000,NaN,NaN,3.0,4.0,5.0,NaN,NaN,NaN
U009,NaN,NaN,NaN,4.0,4.0,3.0,NaN,3.5,4.0,NaN


### Interaction Matrix Interpretation

The user-item matrix contains 50 rows representing the unique users and 24 columns representing the unique movies. Therefore, there are 1,200 possible user-movie interactions.

Many cells contain NaN values because users have only rated some of the available movies. These missing interactions are expected in recommender-system datasets and are used when measuring the sparsity of the interaction matrix.

## 5. Sparsity Calculation

Sparsity measures the proportion of possible user-movie interactions that have not been observed.

The formula used is:

**Sparsity = 1 - (Observed Ratings / (Number of Users × Number of Movies))**

In [16]:
observed_ratings = user_item_matrix.notna().sum().sum()
possible_interactions = unique_users * unique_movies

sparsity = 1 - (observed_ratings / possible_interactions)

print("Observed ratings:", observed_ratings)
print("Possible interactions:", possible_interactions)
print("Sparsity:", round(sparsity, 4))
print("Sparsity percentage:", f"{sparsity * 100:.2f}%")

Observed ratings: 430
Possible interactions: 1200
Sparsity: 0.6417
Sparsity percentage: 64.17%


### Sparsity Interpretation

The user-item interaction matrix has a sparsity of approximately 64.17%. This means that 64.17% of all possible user-movie interactions are missing because those users have not rated those movies.

There are 50 users and 24 movies, giving 1,200 possible user-movie interactions. Of these, 430 unique user-movie interactions are observed in the matrix.

Although the cleaned dataset contains 500 rating records, some users rated the same movie more than once. These repeated ratings are combined into a single user-movie entry when constructing the interaction matrix. Therefore, the matrix contains 430 observed interactions rather than 500.

## 6. Popularity-Based Recommender

A popularity-based recommender suggests movies based on their overall popularity across all users rather than individual user preferences.

For each movie, I calculate two measures:

- **Average rating** – the mean rating given to the movie.
- **Number of ratings** – the total number of ratings the movie has received.

To avoid recommending movies based on very few ratings, only movies with at least 10 ratings are considered for the final recommendations.

In [17]:
movie_stats = (
    df_clean.groupby(["movie_id", "movie_title"])
    .agg(
        average_rating=("rating", "mean"),
        number_of_ratings=("rating", "count")
    )
    .reset_index()
)

movie_stats["average_rating"] = movie_stats["average_rating"].round(2)

movie_stats

,movie_id,movie_title,average_rating,number_of_ratings
0,M001,Nairobi Nights,2.81,37
1,M002,Savannah Run,2.94,34
2,M003,The Last Matatu,3.24,34
3,M004,Coastal Dreams,3.67,27
4,M005,Mountain Echoes,3.86,37
5,M006,City of Tomorrow,3.91,34
6,M007,Hidden Lake,4.12,16
7,M008,The Long Journey,2.60,20
8,M009,Digital Hearts,3.00,15
9,M010,Beyond the Horizon,3.47,15


### Minimum Rating Threshold

Movies with fewer than 10 ratings are excluded from the final recommendation list. This threshold helps reduce the chance of a movie appearing at the top simply because it received a high rating from only a small number of users.

In [18]:
popular_movies = movie_stats[
    movie_stats["number_of_ratings"] >= 10
].copy()

print("Movies with at least 10 ratings:", len(popular_movies))

popular_movies

Movies with at least 10 ratings: 24


,movie_id,movie_title,average_rating,number_of_ratings
0,M001,Nairobi Nights,2.81,37
1,M002,Savannah Run,2.94,34
2,M003,The Last Matatu,3.24,34
3,M004,Coastal Dreams,3.67,27
4,M005,Mountain Echoes,3.86,37
5,M006,City of Tomorrow,3.91,34
6,M007,Hidden Lake,4.12,16
7,M008,The Long Journey,2.60,20
8,M009,Digital Hearts,3.00,15
9,M010,Beyond the Horizon,3.47,15


### Top 10 Recommended Movies

The eligible movies are ranked primarily by their average rating. The number of ratings is used as a secondary sorting criterion when movies have the same average rating.

In [19]:
top_10_movies = (
    popular_movies
    .sort_values(
        by=["average_rating", "number_of_ratings"],
        ascending=[False, False]
    )
    .head(10)
    .reset_index(drop=True)
)

top_10_movies.index = top_10_movies.index + 1

top_10_movies

,movie_id,movie_title,average_rating,number_of_ratings
1,M021,Market Day,4.47,15
2,M013,Midnight in Mombasa,4.35,17
3,M014,Code Breakers,4.25,16
4,M019,Future Africa,4.13,15
5,M020,Home Again,4.13,15
6,M007,Hidden Lake,4.12,16
7,M006,City of Tomorrow,3.91,34
8,M005,Mountain Echoes,3.86,37
9,M012,The Final Goal,3.76,17
10,M004,Coastal Dreams,3.67,27


### Export Recommendations

The final top 10 recommendations are exported to a CSV file so that they can be included separately in the GitHub repository.

In [20]:
top_10_movies.to_csv(
    "../output/top_10_recommendations.csv",
    index=False
)

print("Top 10 recommendations saved successfully.")

Top 10 recommendations saved successfully.


## 7. Interpretations

### Why These Movies Appear at the Top

- The recommended movies appear at the top because they have the highest average ratings among movies that received at least 10 ratings. The minimum threshold helps prevent movies with only a few ratings from ranking highly based on a very small number of users.

Therefore, the recommendation list considers both the overall rating of a movie and whether it has received enough ratings to be included in the ranking.

### Advantage of Popularity-Based Recommendation

- One advantage of a popularity-based recommender is that it is simple to implement and does not require detailed information about individual users. It can provide useful recommendations even for new users who have not yet provided any ratings.

### Limitations

- The first limitation is that the recommendations are not personalized. Every user receives the same top movies regardless of their individual interests or previous ratings.

- The second limitation is that popular movies can dominate the recommendations, making it difficult for less popular or newer movies to appear even when they may be relevant to a particular user.

## 8. Conclusion

- I cleaned and explored a movie-rating dataset and developed a simple popularity-based recommender system.

- After cleaning, the dataset contained 500 valid rating records from 50 users across 24 movies. The user-item interaction matrix contained 430 observed user-movie interactions out of 1,200 possible interactions, resulting in a sparsity of approximately 64.17%.

- The popularity-based recommender ranked movies according to their average ratings while requiring each movie to have at least 10 ratings. This provides a simple baseline recommender that can later be compared with more personalized recommendation techniques.